# Customer Purchase Analytics — Python Analysis
**Python | Pandas | Exploratory Data Analysis | Feature Engineering**

Goal: turn raw customer-shopping data into analysis-ready data and identify customer, product and revenue patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

df = pd.read_csv('customer_shopping_behavior.csv')
print('Rows, columns:', df.shape)
df.head()

## 1. Basic Data Understanding
Before performing analysis, we first inspect the structure, size, data types, missing values and basic statistics of the dataset.

In [ ]:
# First look at the dataset
df.head()

In [ ]:
# View the last few records
df.tail()

In [ ]:
# Number of rows and columns
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

In [ ]:
# Column names
print(df.columns.tolist())

In [ ]:
# Dataset information: columns, data types and non-null values
df.info()

In [ ]:
# Summary statistics for numerical columns
display(df.describe().round(2))

In [ ]:
# Missing-value check
display(df.isnull().sum().sort_values(ascending=False).to_frame('Missing Values'))

In [ ]:
# Duplicate-row check
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
# Number of unique values in important categorical columns
for col in ['Category', 'Gender', 'Season', 'Payment Method']:
    print(f'\n{col}: {df[col].nunique()} unique values')
    print(df[col].value_counts().head())

## 2. Basic Exploratory Analysis
These simple aggregations establish the main business metrics before moving to intermediate analysis.

In [ ]:
# Basic business metrics
total_revenue = df['Purchase Amount (USD)'].sum()
average_purchase = df['Purchase Amount (USD)'].mean()
total_orders = df.shape[0]
unique_customers = df['Customer ID'].nunique()

print(f'Total Revenue: ${total_revenue:,.2f}')
print(f'Average Purchase: ${average_purchase:,.2f}')
print(f'Total Orders: {total_orders:,}')
print(f'Unique Customers: {unique_customers:,}')

In [ ]:
# Revenue by category
category_revenue = df.groupby('Category')['Purchase Amount (USD)'].sum().sort_values(ascending=False)
display(category_revenue.to_frame('Revenue').round(2))

In [ ]:
# Average purchase by gender
gender_purchase = df.groupby('Gender')['Purchase Amount (USD)'].mean().sort_values(ascending=False)
display(gender_purchase.to_frame('Average Purchase').round(2))

In [ ]:
# Most frequently purchased products
top_products = df['Item Purchased'].value_counts().head(10)
display(top_products.to_frame('Purchase Count'))

## 3. Data Quality

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
print('\nMissing values:')
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))
print('\nData types:')
display(df.dtypes.to_frame('dtype'))

In [ ]:
# Standardize text columns and remove exact duplicates
text_cols = df.select_dtypes(include='object').columns
for col in text_cols:
    df[col] = df[col].astype('string').str.strip()

df = df.drop_duplicates().copy()

df['Purchase Amount (USD)'] = pd.to_numeric(df['Purchase Amount (USD)'], errors='coerce')
df['Review Rating'] = pd.to_numeric(df['Review Rating'], errors='coerce')
df['Previous Purchases'] = pd.to_numeric(df['Previous Purchases'], errors='coerce')

# Median is used for missing review ratings because it is less affected by extreme values.
df['Review Rating'] = df['Review Rating'].fillna(df['Review Rating'].median())
print('Clean shape:', df.shape)

## 4. Descriptive Statistics

In [ ]:
display(df[['Age','Purchase Amount (USD)','Review Rating','Previous Purchases']].describe().round(2))

## 5. Business Feature Engineering

In [ ]:
df['Age Group'] = pd.cut(df['Age'], bins=[0,24,34,44,54,64,100], labels=['18-24','25-34','35-44','45-54','55-64','65+'], include_lowest=True)

df['Purchase Band'] = pd.cut(df['Purchase Amount (USD)'], bins=[-np.inf,40,70,np.inf], labels=['Low','Medium','High'])

df['Customer Value Segment'] = np.select(
    [df['Previous Purchases'] >= 35, df['Previous Purchases'] >= 15],
    ['High Value','Growing'],
    default='New / Low Value'
)

df['Discount Flag'] = np.where(df['Discount Applied'].astype('string').str.lower().eq('yes'), 'Discounted', 'Full Price')
df['Repeat Buyer Flag'] = np.where(df['Previous Purchases'].fillna(0) > 0, 'Repeat', 'First Purchase')

df['Rating Band'] = pd.cut(df['Review Rating'], bins=[-np.inf,3,4,np.inf], labels=['Needs Attention','Positive','Excellent'])

frequency_map = {
    'Weekly':'High Frequency', 'Bi-Weekly':'High Frequency', 'Fortnightly':'High Frequency',
    'Monthly':'Medium Frequency', 'Quarterly':'Medium Frequency', 'Every 3 Months':'Medium Frequency',
    'Annually':'Low Frequency'
}
df['Frequency Group'] = df['Frequency of Purchases'].map(frequency_map).fillna('Other')

display(df[['Customer ID','Age Group','Purchase Band','Customer Value Segment','Discount Flag','Repeat Buyer Flag','Rating Band','Frequency Group']].head())

## 6. Category Performance

In [ ]:
category_summary = (df.groupby('Category', as_index=False)
    .agg(Orders=('Customer ID','count'), Revenue=('Purchase Amount (USD)','sum'),
         AOV=('Purchase Amount (USD)','mean'), Avg_Rating=('Review Rating','mean'))
    .sort_values('Revenue', ascending=False))
category_summary.round(2)

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(category_summary['Category'], category_summary['Revenue'])
plt.title('Revenue by Category')
plt.ylabel('Revenue (USD)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 7. Customer-Level Analysis

In [ ]:
customer_summary = (df.groupby('Customer ID', as_index=False)
    .agg(Orders=('Customer ID','count'),
         Total_Spend=('Purchase Amount (USD)','sum'),
         Avg_Order_Value=('Purchase Amount (USD)','mean'),
         Avg_Rating=('Review Rating','mean'),
         Previous_Purchases=('Previous Purchases','max')))

customer_summary['Customer Rank'] = customer_summary['Total_Spend'].rank(method='dense', ascending=False).astype(int)
display(customer_summary.sort_values('Total_Spend', ascending=False).head(10).round(2))

## 8. Pivot Analysis — Category × Season

In [ ]:
season_category = pd.pivot_table(
    df, values='Purchase Amount (USD)', index='Category', columns='Season',
    aggfunc='sum', fill_value=0
)
display(season_category.round(0))

## 9. Discount and Rating Analysis

In [ ]:
discount_summary = (df.groupby('Discount Flag', as_index=False)
    .agg(Orders=('Customer ID','count'), AOV=('Purchase Amount (USD)','mean'), Rating=('Review Rating','mean')))
display(discount_summary.round(2))

## 10. Outlier Review Using IQR

In [ ]:
q1 = df['Purchase Amount (USD)'].quantile(0.25)
q3 = df['Purchase Amount (USD)'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = df[(df['Purchase Amount (USD)'] < lower) | (df['Purchase Amount (USD)'] > upper)]
print(f'IQR lower bound: {lower:.2f}')
print(f'IQR upper bound: {upper:.2f}')
print(f'Potential purchase-value outliers: {len(outliers)}')

## 11. Correlation Check

In [ ]:
numeric_cols = ['Age','Purchase Amount (USD)','Review Rating','Previous Purchases']
display(df[numeric_cols].corr().round(2))

## 12. Export Analysis-Ready Data

In [ ]:
output_file = 'customer_behavior_clean.csv'
df.to_csv(output_file, index=False)
print(f'Saved {output_file} with {len(df):,} rows and {df.shape[1]} columns.')